# Example-11: Save TbT to TEST PVs

In [1]:
# Import

import epics
import numpy
import torch

import sys
sys.path.append('..')

from harmonica.util import pv_make
from harmonica.window import Window
from harmonica.data import Data

import matplotlib.pyplot as plt
from time import sleep

torch.set_printoptions(precision=12, sci_mode=True)
print(torch.cuda.is_available())
print(torch.get_num_threads())

True
16


In [2]:
# Set data type and device

dtype = torch.float64
device = torch.device('cpu')

In [3]:
# Run TEST PVs (softIoc -d harmonica.db)

# Set list of pv names and starting indices

bpm = ['STP2', 'STP4', 'SRP1', 'SRP2', 'SRP3', 'SRP4', 'SRP5', 'SRP6', 'SRP7', 'SRP8', 'SRP9', 'SIP1', 'SIP2', 'SRP10', 'SRP11', 'SRP12', 'SRP13', 'SRP14', 'SRP15', 'SRP16', 'SRP17', 'SEP5', 'SEP4', 'SEP3', 'SEP1', 'SEP0', 'NEP0', 'NEP1', 'NEP3', 'NEP4', 'NEP5', 'NRP17', 'NRP16', 'NRP15', 'NRP14', 'NRP13', 'NRP12', 'NRP11', 'NRP10', 'NIP3', 'NIP1', 'NRP9', 'NRP8', 'NRP7', 'NRP6', 'NRP5', 'NRP4', 'NRP3', 'NRP2', 'NRP1', 'NTP4', 'NTP2', 'NTP0', 'STP0']
pv_list = [pv_make(name, 'X', True) for name in bpm]
pv_rise = [0 for _ in range(len(bpm))]

# Generate test data

frequency = 0.123456
t = torch.linspace(1, 4096, 4096, dtype=dtype, device=device)
data = [(10.0 + torch.randn(1, dtype=dtype, device=device)) * torch.cos(2.0*numpy.pi*frequency*t) for _ in pv_list]
data = torch.stack(data) + 1.0

# Set data 

w = Window(4096, dtype=dtype, device=device)
d = Data.from_data(w, data)
print(w)
print(d)

# Add attibutes

d.pv_list = pv_list
d.pv_rise = pv_rise
d.save_epics()

# Clean

del w
del d
if device != torch.device('cpu'):
    torch.cuda.synchronize()
    torch.cuda.empty_cache()

Window(4096, None, None)
Data(54, Window(4096, None, None))
